In [ ]:
%matplotlib widget
from matplotlib import animation
import numpy as np
import matplotlib.pyplot as plt
import time
from IPython.display import HTML

plt.rcParams['animation.embed_limit'] = 2**128

class Mass(object):
    """
    Class storing information about a mass, including position, and velocity. 

    Data Attributes:
        mass
        position
        velocity
   
    """
    def __init__(self, mass, position, velocity):
        self.mass = mass
        self.position = np.array(position)
        self.velocity = np.array(velocity)

class System():
    def __init__(self, Masses: list[Mass], Natural_units = True, epsilon=0.01):
        self.N = len(Masses)
        self.sys = Masses
        self.epsilon = epsilon

        self.masses = np.array([m.mass for m in Masses])
        self.positions = np.array([m.position for m in Masses])
        self.velocities = np.array([m.velocity for m in Masses])

        if Natural_units:
            self.G = 4 * np.pi * np.pi
        else:
            self.G = 6.67430e-11

        self.trajectory = None

    def acc(self, X, t):
        """This function calculates acceleration using Newtons law of gravitation, given the positions, X of N masses"""

        atot = np.zeros((self.N,3))

        for i in range(self.N):

            for j in range(self.N):
                if j == i:
                    continue

                r_vec = X[j] - X[i] #Calculate displacement between mass i and j
                r = np.linalg.norm(r_vec)

                if r > 0: #epsilon should stop this from breaking (may be redundant)

                    #acceleration of mass i due to mass j
                    a_ij = self.G * self.masses[j] * r_vec / (r**2 + self.epsilon**2)**(3/2)

                    atot[i] += a_ij

        return atot
    
    def acc_2(self, X, t):
        """This function calculates acceleration using Newtons law of gravitation, given the positions, X of N masses"""
        #Does vectorized operations, should be faster for larger N, but slower for small N due to overhead of vectorization
        
        X_i = X[:, np.newaxis, :]  # (N, 1, 3)
        X_j = X[np.newaxis, :, :]  # (1, N, 3)
        r_vec = X_j - X_i  # (N, N, 3)

        r = np.linalg.norm(r_vec, axis=2)  # (N, N)
        r = np.where(r == 0, np.inf, r)
        acc_matrix = self.G * self.masses[np.newaxis, :, np.newaxis] * r_vec / (r[:, :, np.newaxis]**2 + self.epsilon**2)**(3/2)
        atot = np.sum(acc_matrix, axis=1)  #shape (N, 3)

        return atot
    
    def rk4(self, t, acc_func = None, bump = False, bumpfreq = 10, bumpamount = 1.01):
        """This function solves the motion of a system of masses using the Runge-Kutta 4th order method"""
        dt = t[1] - t[0]   

        if acc_func is None:
            #Currently have 2 acceleration functions. Can choose or defaults to acc_2
            acc_func = self.acc_2

        x = self.positions
        v = self.velocities

        k1v  = dt * acc_func(x, t[0])
        k1x = dt * v

        k2v = dt * acc_func(x + k1x/2, t[0] + dt/2)
        k2x = dt * (v + k1v/2) 

        k3v = dt * acc_func(x + k2x/2, t[0] + dt/2)
        k3x = dt * (v + k2v/2)

        k4v = dt * acc_func(x + k3x, t[0] + dt)
        k4x = dt * (v + k3v)

        #Allocating space for trajectory
        xtraj = np.zeros((len(t), self.N, 3))
        vtraj = np.zeros((len(t), self.N, 3))
        
        #Setting initial position and velocity
        xtraj[0] = x
        vtraj[0] = v

        for i in range(1, len(t)):
            term2x = 1/6 * (k1x + 2*k2x + 2*k3x + k4x)
            term2v = 1/6 * (k1v + 2*k2v + 2*k3v + k4v)

            x = x + term2x                                                                  ##CHANGE THE VALUES HERE MAYBE? and leapfrog
            v = v + term2v
            if bump and i%bumpfreq == 0 and i != 0:
                v[-1] *= bumpamount

            self.positions = x
            self.velocities = v
            
            k1v  = dt * acc_func(x, t[i])
            k1x = dt * v

            k2v = dt * acc_func(x + k1x/2, t[i] + dt/2)
            k2x = dt * (v + k1v/2) 

            k3v = dt * acc_func(x + k2x/2, t[i] + dt/2)
            k3x = dt * (v + k2v/2)

            k4v = dt * acc_func(x + k3x, t[i] + dt)
            k4x = dt * (v + k3v)

            xtraj[i] = x
            vtraj[i] = v

        #Combine into a single array of shape (len(t), N, 6)
        ptraj = np.concatenate((xtraj, vtraj), axis = 2)
        self.trajectory = ptraj

        return ptraj
    
    def leapfrog(self, t, acc_func = None, bump = False, bumpfreq = 10, bumpamount = 1.01):
        """This function solves the motion of a system of masses using the leapfrog method."""
        dt = t[1] - t[0]

        x = self.positions
        v = self.velocities 

        if acc_func is None:
            #Currently have 2 acceleration functions, can choose, or defaults to acc_2
            acc_func = self.acc_2

        a = acc_func(x, t[0])

        #first half step for velocity, using acceleration at initial positions
        v_half = v + a * dt / 2

        #Allocating space for trajectory
        xtraj = np.zeros((len(t), self.N, 3))
        vtraj = np.zeros((len(t), self.N, 3))

        #Setting initial position and velocity (initial v_half)
        xtraj[0] = x
        vtraj[0] = v_half

        for i in range(1, len(t)):

            x = x + v_half * dt

            a = acc_func(x, t)
        
            v_half = v_half + a * dt

            if bump and i%bumpfreq == 0 and i != 0:
                v_half[-1] *= bumpamount

            self.positions = x
            self.velocities = v_half

            xtraj[i] = x
            vtraj[i] = v_half

        #Combine into a single array of shape (len(t), N, 6)
        ptraj = np.concatenate((xtraj, vtraj), axis = 2)

        self.trajectory = ptraj

        return ptraj
    
    
    def earthcentric(self):
        """"This function transforms the trajectory to be centered on the first body (assumed to be Earth)"""
        N = len(self.trajectory)
        for i in range(N):
            positions = self.trajectory[i, 0, :3]
            self.trajectory[i, :, :3] = self.trajectory[i, :, :3] - positions
    
    
    def plot(self, elev = 90, azim = -90):
        """This function plots a trajectory using matplotlib in 3d. The view angle can be altered using elev and azim input parameters"""
        if self.trajectory is None:
            raise ValueError("No trajectory found. Run rk4 or leapfrog to generate trajectory before plotting.")
        
        fig, ax = plt.subplots(subplot_kw={'projection': '3d'})
        ax.set_xlabel('X(AU)')
        ax.set_ylabel('Y(AU)')
        ax.set_zlabel('Z(AU)')
        ax.view_init(elev=elev, azim=azim)

        positions = self.trajectory[:,:,:3]

        for i in range(self.N):
            ax.plot(positions[:,i,0], positions[:,i,1], positions[:,i,2])
        plt.show()

    def _init_animate(self):
        """This function sets the initial objects to be animated"""
        for line, pt in zip(self.lines, self.pts):
            line.set_data([], [])
            line.set_3d_properties([])

            pt.set_data([], [])
            pt.set_3d_properties([])

        return self.lines + self.pts
    
    def _Animate(self, itr):
        """This function draws lines and points for the masses in the system up to index itr."""
        #shape (tmax, N, 3)
        positions = self.trajectory[:,:,:3]

        #shape (N, tmax, 3)
        positions = positions.transpose(1, 0, 2)

        for line, pt, pi in zip(self.lines, self.pts, positions):

            line.set_data(pi[:itr,0].T, pi[:itr,1].T)
            line.set_3d_properties(pi[:itr,2].T)

            pt.set_data([pi[itr,0].T], [pi[itr,1].T])
            pt.set_3d_properties([pi[itr,2].T])

        return self.lines + self.pts
    
    def Simulate(self, title = 'N-body Simulation', save_as = None, display = True, fpss = 30, xlim = None, ylim = None, pointsize = 6.0, elev = None, azim = None):
        """Animates using Matplotlib"""

        if self.trajectory is None:
            #may just run one of them automatically later, instead of raising an error
            raise ValueError("No trajectory found. Run rk4 or leapfrog to generate trajectory before simulating.")

        fig, ax = plt.subplots(subplot_kw={'projection': '3d'}, figsize = (12, 12))
        ax.set_title(f"{title}")
        ax.set_xlabel('X(AU)')
        ax.set_ylabel('Y(AU)')
        ax.set_zlabel('Z(AU)')
        if xlim is None:
            ax.set_xlim(np.min(self.trajectory[:,:,0]), np.max(self.trajectory[:,:,0]))
        else:
            ax.set_xlim(xlim)
        if ylim is None:
            ax.set_ylim(np.min(self.trajectory[:,:,1]), np.max(self.trajectory[:,:,1]))
        else:
            ax.set_ylim(ylim)
        ax.view_init(elev=elev, azim=azim)

        
        z_min, z_max = np.min(self.trajectory[:,:,2]), np.max(self.trajectory[:,:,2])
        if z_min == z_max:
            ax.set_zlim(z_min - 0.1, z_max + 0.1)
        else:
            ax.set_zlim(z_min, z_max)

        colors = plt.cm.jet(np.linspace(0, 1, self.N))

        self.lines = sum([ax.plot([], [], [], '-', c=c)
             for c in colors], [])
        
        self.pts = sum([ax.plot([], [], [], 'o', ms = pointsize, c=c)
           for c in colors], [])
        

        anim = animation.FuncAnimation(fig, self._Animate, init_func = self._init_animate, frames=len(self.trajectory), interval=1000/fpss)

        if save_as is not None:
            anim.save(save_as, writer='pillow', fps=fpss)
        
        if display:
            display(HTML(anim.to_jshtml())) 

    def energy_cons(self):
        """Compares initial to final energy to check conservation of energy.
        Returns the ratio of initial to final energy, which should be close to 1 for good energy conservation."""
        
        if self.trajectory is None:
            raise ValueError("No trajectory found. Run rk4 or leapfrog to generate trajectory before checking energy conservation.")
        
        initial_positions = self.trajectory[0,:,:3]
        initial_velocities = self.trajectory[0,:,3:]

        final_positions = self.trajectory[-1,:,:3]
        final_velocities = self.trajectory[-1,:,3:]

        #Kinetic energy: K = 1/2 m v^2
        K_initial = 0.5 * np.sum(self.masses * np.sum(initial_velocities**2, axis=1))
        K_final = 0.5 * np.sum(self.masses * np.sum(final_velocities**2, axis=1))

        #Potential energy: U = -G * sum(m_i * m_j / sqrt(r_ij^2 + epsilon^2))
        U_initial = 0
        U_final = 0

        for i in range(self.N):
            for j in range(i + 1, self.N):
                r_ij = np.linalg.norm(initial_positions[i] - initial_positions[j])
                U_initial -= self.G * self.masses[i] * self.masses[j] / np.sqrt(r_ij**2 + self.epsilon**2)

                r_ij = np.linalg.norm(final_positions[i] - final_positions[j])
                U_final -= self.G * self.masses[i] * self.masses[j] / np.sqrt(r_ij**2 + self.epsilon**2)

        # Total energy
        E_initial = K_initial + U_initial
        E_final = K_final + U_final

        print(f"Ei / Ef: {E_initial / E_final}")

        return E_initial / E_final
        
        

In [ ]:
#Galaxy Collision

def pctoau(pc):
    """
    Calculates the distance in astronomical Units given a distance in parsecs

    Parameters:
    pc - distance in parsecs
    """
    return pc*206265

def period(a, M):   ######IN NATURAL UNITS
    """
    Calculates the period of an orbit
    All units are in natural units of AU, Solar Masses, and Years
    
    Parameters:
    a - semimajor axis
    M - total system mass
    """
    return a**(3/2)/M**(1/2)

def escapevelocity(x, b, M1, M2):
    """
    Calculates the escape velocity of a system
    All units are in natural units of AU, Solar Masses, and Years
    
    Parameters:
    x - x distance from the body at rest
    b - impact parameter (y distance from body at rest)
    M1 - mass of the body at rest
    M2 - mass of the moving body
    """
    r = np.sqrt(x*x+b*b)
    return np.sqrt(2*4*np.pi*np.pi*(M1+M2)/r)

def creategalaxysystem(b, innerringstars = 10, retrograde = False):
    """
    Creates a System object representing a galaxy with 3 rings of infinitesimal mass stars around it and a perturber (colliding) galaxy
    All units are in natural units of AU, Solar Masses, and Years
    
    Parameters:
    b - impact parameter
    innerringstars - number of stars in the innerring of the main galaxy. The middle ring has twice as many, and the outer ring has 3 times as many (default = 10)
    retrograde - whether the rings of stars are in retrograde relative to the perturber galaxy (default = False)
    """
    
    system = []
    mainmass = 10**11
    perturmass = 10**10
    main = Mass(mainmass, np.zeros(3), np.zeros(3))

    xinitper = np.sqrt(pctoau(100000.0)**2-b*b)                                                         #Calculates the x distance of the perturber to the main galaxy, so total distance is 100kpc
    
    vinitper = escapevelocity(xinitper, b, mainmass, perturmass)                                        #Perturper speed, escape velocity, E = 0
    perturber = Mass(perturmass, np.array([xinitper, b, 0.0]), np.array([-vinitper, 0.0, 0.0]))
    
    system.append(main)
    system.append(perturber)

    innerring = np.linspace(0, 2*np.pi*(innerringstars-1)/innerringstars, innerringstars)               #Arrays of angles where a star appears in each ring
    midring = np.linspace(0, 2*np.pi*(2*innerringstars-1)/2/innerringstars, 2*innerringstars)
    outerring = np.linspace(0, 2*np.pi*(3*innerringstars-1)/3/innerringstars, 3*innerringstars)

    ri = pctoau(5000)
    rm = pctoau(10000)
    ro = pctoau(15000)

    ip = period(ri, mainmass)  #Inner ring period
    mp = period(rm, mainmass) #Mid ring period
    op = period(ro, mainmass) #Outer ring period

    vi = 2*np.pi*ri/ip            #Inner ring speed
    vm = 2*np.pi*rm/mp            #Mid ring speed
    vo = 2*np.pi*ro/op            #Outer ring speed

    
    
    for i in innerring:                                                                                                 #Creating a massless Mass object at each angle, appening to the system
        ringstar = Mass(0, np.array([ri*np.cos(i), ri*np.sin(i), 0]), np.array([-vi*np.sin(i), vi*np.cos(i), 0]))
        if retrograde:
            ringstar.velocity *= (-1)
        system.append(ringstar)

    for i in midring:
        ringstar = Mass(0, np.array([rm*np.cos(i), rm*np.sin(i), 0]), np.array([-vm*np.sin(i), vm*np.cos(i), 0]))
        if retrograde:
            ringstar.velocity *= (-1)
        system.append(ringstar)

    for i in outerring:
        ringstar = Mass(0, np.array([ro*np.cos(i), ro*np.sin(i), 0]), np.array([-vo*np.sin(i), vo*np.cos(i), 0]))
        if retrograde:
            ringstar.velocity *= (-1)
        system.append(ringstar)

    return System(system)



In [ ]:
# Three Body

def threebody(M1 = 1.0, M2 = 1.0, M3 = 1.0, separation = 1.0, M3velocity = 10, M3dist = 10, b = 0):
    """
    Creates a System object representing a three body system with 2 bodies in a stable orbit and a third object perturbing the system
    All units are in natural units of AU, Solar Masses, and Years

    Parameters:
    M1 - mass of one of the orbiting bodies (default = 1.0)
    M2 - mass of the other orbiting body (default = 1.0)
    M3 - mass of the perturbing body (default = 1.0)
    separation - inital separation of the orbiting bodies, assuming this separation occurs at aphelion of both orbits (default = 1.0)
    M3 - initial velocity of the perturbing body (default = 10.0)
    M3dist - initial distance of the perturbing body from the origin, M1 (default = 10)
    b - impact parameter
    """
    COM = M2*separation/(M1+M2)    #Calculating Center of Mass with Origin at M1 
    M1dist = -COM                  #Shifting Origin to the Center of Mass
    M2dist = separation-COM
    orbitalperiod = period(abs(M1dist), (M1+M2))
    body1 = Mass(M1, np.array([M1dist, 0.0, 0.0]), np.array([0.0, -2*np.pi*abs(M1dist)/orbitalperiod, 0.0]))   #These initial conditions are at aphelion.
    body2 = Mass(M2, np.array([M2dist, 0.0, 0.0]), np.array([0.0, 2*np.pi*abs(M2dist)/orbitalperiod, 0.0]))
    body3 = Mass(M3, np.array([M3dist, b, 0.0]), np.array([-M3velocity, 0.0, 0.0]))
    return System([body1, body2, body3])


In [ ]:
#Galaxy Collision Analysis 1 (b = 20000)

galsys1 = creategalaxysystem(pctoau(20000.0))

galtraj1 = galsys1.rk4(np.linspace(0, 10**9, 500))

galsys1.Simulate('Galaxy Collision with b = 20 kpc - RK4', save_as="Galaxy Collision 20 RK4.gif", fpss = 15, xlim = (-pctoau(30000),pctoau(30000)), ylim = (-pctoau(30000),pctoau(30000)))



In [ ]:
#Galaxy Collision Analysis 1 (b = 20000)

galsys1l = creategalaxysystem(pctoau(20000.0))

galtraj1l = galsys1l.leapfrog(np.linspace(0, 10**9, 500))

galsys1l.Simulate('Galaxy Collision with b = 20 kpc - Leapfrog', save_as="Galaxy Collision 20 Leapfrog.gif", fpss = 15, xlim = (-pctoau(30000),pctoau(30000)), ylim = (-pctoau(30000),pctoau(30000)))


In [ ]:
#Galaxy Collision Analysis 2 (b = 30000)

galsys2 = creategalaxysystem(pctoau(30000.0))

galtraj2 = galsys2.rk4(np.linspace(0, 10**9, 500))

galsys2.Simulate('Galaxy Collision with b = 30 kpc - RK4', save_as="Galaxy Collision 30 RK4.gif", fpss = 15, xlim = (-pctoau(35000),pctoau(35000)), ylim = (-pctoau(35000),pctoau(35000)))

#Galaxy Collision Analysis 2 (b = 30000)

galsys2l = creategalaxysystem(pctoau(30000.0))

galtraj2l = galsys2l.leapfrog(np.linspace(0, 10**9, 500))

galsys2l.Simulate('Galaxy Collision with b = 30 kpc - Leapfrog', save_as="Galaxy Collision 30 Leapfrog.gif", fpss = 15, xlim = (-pctoau(35000),pctoau(35000)), ylim = (-pctoau(35000),pctoau(35000)))


In [ ]:
#Galaxy Collision Analysis 3 (b = 40000)

galsys3 = creategalaxysystem(pctoau(40000.0))

galtraj3 = galsys3.rk4(np.linspace(0, 10**9, 500))

galsys3.Simulate('Galaxy Collision with b = 40 kpc - RK4', save_as="Galaxy Collision 40 RK4.gif", fpss = 15, xlim = (-pctoau(45000),pctoau(45000)), ylim = (-pctoau(45000),pctoau(45000)))

#Galaxy Collision Analysis 3 (b = 40000)

galsys3l = creategalaxysystem(pctoau(40000.0))

galtraj3l = galsys3l.leapfrog(np.linspace(0, 10**9, 500))

galsys3l.Simulate('Galaxy Collision with b = 40 kpc - Leapfrog', save_as="Galaxy Collision 40 Leapfrog.gif", fpss = 15, xlim = (-pctoau(45000),pctoau(45000)), ylim = (-pctoau(45000),pctoau(45000)))


In [ ]:
#Galaxy Collision Analysis 4 (b = 30000, reverse)

galsys4 = creategalaxysystem(pctoau(30000.0), retrograde= True)

galtraj4 = galsys4.rk4(np.linspace(0, 10**9, 500))

galsys4.Simulate('Galaxy Collision with b = 30 kpc and Retrograde Motion - RK4', save_as="Galaxy Collision 30 Retrograde RK4.gif", fpss = 15, xlim = (-pctoau(35000),pctoau(35000)), ylim = (-pctoau(35000),pctoau(35000)))

#Galaxy Collision Analysis 4 (b = 30000, reverse)

galsys4l = creategalaxysystem(pctoau(30000.0), retrograde= True)

galtraj4l = galsys4l.leapfrog(np.linspace(0, 10**9, 500))

galsys4l.Simulate('Galaxy Collision with b = 30 kpc and Retrograde Motion - Leapfrog', save_as="Galaxy Collision 30 Retrograde Leapfrog.gif", fpss = 15, xlim = (-pctoau(35000),pctoau(35000)), ylim = (-pctoau(35000),pctoau(35000)))


In [ ]:
sun = Mass(1, np.zeros(3), np.zeros(3))
earth = Mass(3.00274e-6, np.array([1,0,0]), np.array([0,2*np.pi,0]))

In [ ]:
#Lagrange Analysis 1 - No change to L2 and L4
L2 = Mass(0, np.array([1.01,0,0]), np.array([0,2*np.pi*1.01/period(1.01, 1+3.00274e-6),0]))

L2sys1 = System([earth, sun, L2])
L2traj1 = L2sys1.rk4(np.linspace(0, 1, 1000))
L2sys1.earthcentric()

L2sys1.Simulate('Earth-Sun L2 Point System - RK4', save_as="L2 point RK4.gif", fpss = 15)



L4 = Mass(0, np.array([np.cos(np.pi/3), np.sin(np.pi/3), 0]), np.array([-2*np.pi*np.sin(np.pi/3), 2*np.pi*np.cos(np.pi/3),0]))

L4sys1 = System([earth, sun, L4])
L4traj1 = L4sys1.rk4(np.linspace(0, 1, 1000))
L4sys1.earthcentric()

L4sys1.Simulate('Earth-Sun L4 Point System - RK4', save_as="L4 point RK4.gif", fpss = 15)


L2 = Mass(0, np.array([1.01,0,0]), np.array([0,2*np.pi*1.01/period(1.01, 1+3.00274e-6),0]))

L2sys1l = System([earth, sun, L2])
L2traj1l = L2sys1l.leapfrog(np.linspace(0, 1, 1000))
L2sys1l.earthcentric()

L2sys1l.Simulate('Earth-Sun L2 Point System - Leapfrog', save_as="L2 point Leapfrog.gif", fpss = 15)



L4 = Mass(0, np.array([np.cos(np.pi/3), np.sin(np.pi/3), 0]), np.array([-2*np.pi*np.sin(np.pi/3), 2*np.pi*np.cos(np.pi/3),0]))

L4sys1l = System([earth, sun, L4])
L4traj1l = L4sys1l.leapfrog(np.linspace(0, 1, 1000))
L4sys1l.earthcentric()

L4sys1l.Simulate('Earth-Sun L4 Point System - Leapfrog', save_as="L4 point Leapfrog.gif", fpss = 15)


In [ ]:
#Lagrange Analysis 2 - Small semi frequent add to L2 and L4
L2 = Mass(0, np.array([1.01,0,0]), np.array([0,2*np.pi*1.01/period(1.01, 1+3.00274e-6),0]))

L2sys2 = System([earth, sun, L2])
L2traj2 = L2sys2.rk4(np.linspace(0, 1, 1000), bump = True, bumpamount = 1.01, bumpfreq = 100)
L2sys2.earthcentric()

L2sys2.Simulate('Earth-Sun L2 Point System with 1% Momentum Increase Every Tenth of an Orbit - RK4', save_as="L2 point Increase Tenth RK4.gif", fpss = 15)



L4 = Mass(0, np.array([np.cos(np.pi/3), np.sin(np.pi/3), 0]), np.array([-2*np.pi*np.sin(np.pi/3), 2*np.pi*np.cos(np.pi/3),0]))

L4sys2 = System([earth, sun, L4])
L4traj2 = L4sys2.rk4(np.linspace(0, 1, 1000), bump = True, bumpamount = 1.01, bumpfreq = 100)
L4sys2.earthcentric()

L4sys2.Simulate('Earth-Sun L4 Point System with 1% Momentum Increase Every Tenth of an Orbit - RK4', save_as="L4 point Increase Tenth RK4.gif", fpss = 15)

#Lagrange Analysis 2 - Small semi frequent add to L2 and L4

L2 = Mass(0, np.array([1.01,0,0]), np.array([0,2*np.pi*1.01/period(1.01, 1+3.00274e-6),0]))

L2sys2l = System([earth, sun, L2])
L2traj2l = L2sys2l.leapfrog(np.linspace(0, 1, 1000), bump = True, bumpamount = 1.01, bumpfreq = 100)
L2sys2l.earthcentric()

L2sys2l.Simulate('Earth-Sun L2 Point System with 1% Momentum Increase Every Tenth of an Orbit - Leapfrog', save_as="L2 point Increase Tenth Leapfrog.gif", fpss = 15)



L4 = Mass(0, np.array([np.cos(np.pi/3), np.sin(np.pi/3), 0]), np.array([-2*np.pi*np.sin(np.pi/3), 2*np.pi*np.cos(np.pi/3),0]))

L4sys2l = System([earth, sun, L4])
L4traj2l = L4sys2l.leapfrog(np.linspace(0, 1, 1000), bump = True, bumpamount = 1.01, bumpfreq = 100)
L4sys2l.earthcentric()

L4sys2l.Simulate('Earth-Sun L4 Point System with 1% Momentum Increase Every Tenth of an Orbit - Leapfrog', save_as="L4 point Increase Tenth Leapfrog.gif", fpss = 15)

In [ ]:
#Lagrange Analysis 3 - Small uncommon add to L2
L2 = Mass(0, np.array([1.01,0,0]), np.array([0,2*np.pi*1.01/period(1.01, 1+3.00274e-6),0]))

L2sys3 = System([earth, sun, L2])
L2traj3 = L2sys3.rk4(np.linspace(0, 1, 1001), bump = True, bumpamount = 1.01, bumpfreq = 500)
L2sys3.earthcentric()

L2sys3.Simulate('Earth-Sun L2 Point System with 1% Momentum Increase Every Half of an Orbit - RK4', save_as="L2 point Increase Half RK4.gif", fpss = 15)

L2 = Mass(0, np.array([1.01,0,0]), np.array([0,2*np.pi*1.01/period(1.01, 1+3.00274e-6),0]))

L2sys3l = System([earth, sun, L2])
L2traj3l = L2sys3l.leapfrog(np.linspace(0, 1, 1001), bump = True, bumpamount = 1.01, bumpfreq = 500)
L2sys3l.earthcentric()

L2sys3l.Simulate('Earth-Sun L2 Point System with 1% Momentum Increase Every Half of an Orbit - Leapfrog', save_as="L2 point Increase Half Leapfrog.gif", fpss = 15)


In [ ]:
#Lagrange Analysis 4 - Small semi frequent decrease to L2 and L4

L2 = Mass(0, np.array([1.01,0,0]), np.array([0,2*np.pi*1.01/period(1.01, 1+3.00274e-6),0]))

L2sys4 = System([earth, sun, L2])
L2traj4 = L2sys4.rk4(np.linspace(0, 1, 1000), bump = True, bumpamount = 0.99, bumpfreq = 100)
L2sys4.earthcentric()

L2sys4.Simulate('Earth-Sun L2 Point System with 1% Momentum Decrease Every Tenth of an Orbit - RK4', save_as="L2 point Decrease Tenth RK4.gif", fpss = 15)



L4 = Mass(0, np.array([np.cos(np.pi/3), np.sin(np.pi/3), 0]), np.array([-2*np.pi*np.sin(np.pi/3), 2*np.pi*np.cos(np.pi/3),0]))

L4sys4 = System([earth, sun, L4])
L4traj4 = L4sys4.rk4(np.linspace(0, 1, 1000), bump = True, bumpamount = 0.99, bumpfreq = 100)
L4sys4.earthcentric()

L4sys4.Simulate('Earth-Sun L4 Point System with 1% Momentum Decrease Every Tenth of an Orbit - RK4', save_as="L4 point Decrease Tenth RK4.gif", fpss = 15)

#Lagrange Analysis 4 - Small semi frequent decrease to L2 and L4

L2 = Mass(0, np.array([1.01,0,0]), np.array([0,2*np.pi*1.01/period(1.01, 1+3.00274e-6),0]))

L2sys4l = System([earth, sun, L2])
L2traj4l = L2sys4l.leapfrog(np.linspace(0, 1, 1000), bump = True, bumpamount = 0.99, bumpfreq = 100)
L2sys4l.earthcentric()

L2sys4l.Simulate('Earth-Sun L2 Point System with 1% Momentum Decrease Every Tenth of an Orbit - Leapfrog', save_as="L2 point Decrease Tenth Leapfrog.gif", fpss = 15)



L4 = Mass(0, np.array([np.cos(np.pi/3), np.sin(np.pi/3), 0]), np.array([-2*np.pi*np.sin(np.pi/3), 2*np.pi*np.cos(np.pi/3),0]))

L4sys4l = System([earth, sun, L4])
L4traj4l = L4sys4l.leapfrog(np.linspace(0, 1, 1000), bump = True, bumpamount = 0.99, bumpfreq = 100)
L4sys4l.earthcentric()

L4sys4l.Simulate('Earth-Sun L4 Point System with 1% Momentum Decrease Every Tenth of an Orbit - Leapfrog', save_as="L4 point Decrease Tenth Leapfrog.gif", fpss = 15)

In [ ]:
#Lagrange Analysis 5 - Small uncommon decrease to L2
L2 = Mass(0, np.array([1.01,0,0]), np.array([0,2*np.pi*1.01/period(1.01, 1+3.00274e-6),0]))

L2sys5 = System([earth, sun, L2])
L2traj5 = L2sys5.rk4(np.linspace(0, 1, 1001), bump = True, bumpamount = 0.99, bumpfreq = 500)
L2sys5.earthcentric()

L2sys5.Simulate('Earth-Sun L2 Point System with 1% Momentum Decrease Every Half of an Orbit - RK4', save_as="L2 point Decrease Half RK4.gif", fpss = 15)


#Lagrange Analysis 5 - Small uncommon decrease to L2
L2 = Mass(0, np.array([1.01,0,0]), np.array([0,2*np.pi*1.01/period(1.01, 1+3.00274e-6),0]))

L2sys5l = System([earth, sun, L2])
L2traj5l = L2sys5l.rk4(np.linspace(0, 1, 1001), bump = True, bumpamount = 0.99, bumpfreq = 500)
L2sys5l.earthcentric()

L2sys5l.Simulate('Earth-Sun L2 Point System with 1% Momentum Decrease Every Half of an Orbit - Leapfrog', save_as="L2 point Decrease Half Leapfrog.gif", fpss = 15)


In [ ]:
#Three body Analysis initial orbit energy
twobodyenergy = threebody(M1 = 1, M2 = 15, M3 = 0, separation= 1.0)
print(twobodyenergy.energy())

In [ ]:
#Three Body Analysis 1

tb1 = threebody(M1 = 1, M2 = 15, M3 = 10, separation= 1.0, M3velocity= 10.0, M3dist= 50.0, b = 0.5)
print(tb1.energy())

tb1traj = tb1.rk4(np.linspace(0, 10, 1000))

print(tb1.energy())

tb1.Simulate("Three-body System with M1 = 1, M2 = 15, M3 = 10, M3 Velocity = 10, M3 Position = (50.0, 0.5) - RK4", save_as="Three-body System 1 RK4.gif", fpss = 15, xlim = (-10, 10), ylim = (-10, 10))

#Three Body Analysis 1

tb1l = threebody(M1 = 1, M2 = 15, M3 = 10, separation= 1.0, M3velocity= 10.0, M3dist= 50.0, b = 0.5)
print(tb1l.energy())

tb1trajl = tb1l.leapfrog(np.linspace(0, 10, 1000))

print(tb1l.energy())

tb1l.Simulate("Three-body System with M1 = 1, M2 = 15, M3 = 10, M3 Velocity = 10, M3 Position = (50.0, 0.5) - Leapfrog", save_as="Three-body System 1 Leapfrog.gif", fpss = 15, xlim = (-10, 10), ylim = (-10, 10))


In [ ]:
#Three Body Analysis 2

tb2 = threebody(M1 = 1, M2 = 15, M3 = 10, separation= 1.0, M3velocity= 10.0, M3dist= 50.0, b = 10.0)
print(tb2.energy())

tb2traj = tb2.rk4(np.linspace(0, 10, 1000))

print(tb2.energy())

tb2.Simulate("Three-body System with M1 = 1, M2 = 15, M3 = 10, M3 Velocity = 10, M3 Position = (50.0, 10.0) - RK4", save_as="Three-body System 2 RK4.gif", fpss = 15, xlim = (-10, 10), ylim = (-10, 10))

#Three Body Analysis 2

tb2l = threebody(M1 = 1, M2 = 15, M3 = 10, separation= 1.0, M3velocity= 10.0, M3dist= 50.0, b = 10.0)
print(tb2l.energy())

tb2trajl = tb2l.leapfrog(np.linspace(0, 10, 1000))

print(tb2l.energy())

tb2l.Simulate("Three-body System with M1 = 1, M2 = 15, M3 = 10, M3 Velocity = 10, M3 Position = (50.0, 10.0) - Leapfrog", save_as="Three-body System 2 Leapfrog.gif", fpss = 15, xlim = (-10, 10), ylim = (-10, 10))


In [ ]:
#Three Body Analysis 3

tb3 = threebody(M1 = 1, M2 = 15, M3 = 10, separation= 1.0, M3velocity= 25.0, M3dist= 50.0, b = 0.5)
print(tb3.energy())

tb3traj = tb3.rk4(np.linspace(0, 10, 1000))

print(tb3.energy())

tb3.Simulate("Three-body System with M1 = 1, M2 = 15, M3 = 10, M3 Velocity = 25, M3 Position = (50.0, 0.5) - RK4", save_as="Three-body System 3 RK4.gif", fpss = 15, xlim = (-10, 10), ylim = (-10, 10))

#Three Body Analysis 3

tb3l = threebody(M1 = 1, M2 = 15, M3 = 10, separation= 1.0, M3velocity= 25.0, M3dist= 50.0, b = 0.5)
print(tb3l.energy())

tb3trajl = tb3l.leapfrog(np.linspace(0, 10, 1000))

print(tb3l.energy())

tb3l.Simulate("Three-body System with M1 = 1, M2 = 15, M3 = 10, M3 Velocity = 25, M3 Position = (50.0, 0.5) - Leapfrog", save_as="Three-body System 3 Leapfrog.gif", fpss = 15, xlim = (-10, 10), ylim = (-10, 10))


In [ ]:
#Three Body Analysis 4

tb4 = threebody(M1 = 1, M2 = 15, M3 = 10, separation= 1.0, M3velocity= 25.0, M3dist= 50.0, b = 10.0)
print(tb4.energy())

tb4traj = tb4.rk4(np.linspace(0, 10, 1000))

print(tb4.energy())

tb4.Simulate("Three-body System with M1 = 1, M2 = 15, M3 = 10, M3 Velocity = 25, M3 Position = (50.0, 10.0) - RK4", save_as="Three-body System 4 RK4.gif", fpss = 15, xlim = (-10, 10), ylim = (-10, 10))

#Three Body Analysis 4

tb4l = threebody(M1 = 1, M2 = 15, M3 = 10, separation= 1.0, M3velocity= 25.0, M3dist= 50.0, b = 10.0)
print(tb4l.energy())

tb4trajl = tb4l.leapfrog(np.linspace(0, 10, 1000))

print(tb4l.energy())

tb4l.Simulate("Three-body System with M1 = 1, M2 = 15, M3 = 10, M3 Velocity = 25, M3 Position = (50.0, 10.0) - Leapfrog", save_as="Three-body System 4 Leapfrog.gif", fpss = 15, xlim = (-10, 10), ylim = (-10, 10))


In [ ]:
#Checking Efficiency and Accuracy (time taken by integrator and energy conservation)

#Earth-Sun

es = System([sun,earth])
print("Earth-Sun RK4")
ts = time.time()
ptraj = es.rk4(np.linspace(0,1000,100000))
te = time.time()
print(f"Time: {te - ts:.2f} seconds.")
e1 = es.energy_cons()

es = System([sun,earth])
print("\nEarth-Sun Leapfrog")
ts = time.time()
ptraj2 = es.leapfrog(np.linspace(0,1000,100000))
te = time.time()
print(f"Time: {te - ts:.2f} seconds.")
e2 = es.energy_cons()


#Lagrange

L2sys = System([earth, sun, L2])
print("\nL2 RK4")
ts = time.time()
L2traj = L2sys.rk4(np.linspace(0, 1000, 100000))
te = time.time()
print(f"Time: {te - ts:.2f} seconds.")
e3 = L2sys.energy_cons()

L2sys = System([earth, sun, L2])
print("\nL2 Leapfrog")
ts = time.time()
L2traj = L2sys.leapfrog(np.linspace(0, 1000, 100000))
te = time.time()
print(f"Time: {te - ts:.2f} seconds.")
e4 = L2sys.energy_cons()

L4sys = System([earth, sun, L4])
print("\nL4 RK4")
ts = time.time()
L4traj = L4sys.rk4(np.linspace(0, 1000, 100000))
te = time.time()
print(f"Time: {te - ts:.2f} seconds.")
e5 = L4sys.energy_cons()

L4sys = System([earth, sun, L4])
print("\nL4 Leapfrog")
ts = time.time()
L4traj = L4sys.leapfrog(np.linspace(0, 1000, 100000))
te = time.time()
print(f"Time: {te - ts:.2f} seconds.")
e6 = L4sys.energy_cons()


#Three Body

tb1 = threebody(M1 = 1, M2 = 15, M3 = 10, separation= 1.0, M3velocity= 10.0, b = 0.5)
print("\nThree Body Rk4")
ts = time.time()
tb1traj = tb1.rk4(np.linspace(0, 5, 100000))
te = time.time()
print(f"Time: {te - ts:.2f} seconds.")
e7 = tb1.energy_cons()

tb1 = threebody(M1 = 1, M2 = 15, M3 = 10, separation= 1.0, M3velocity= 10.0, b = 0.5)
print("\nThree Body Leapfrog")
ts = time.time()
tb1traj = tb1.leapfrog(np.linspace(0, 5, 100000))
te = time.time()
print(f"Time: {te - ts:.2f} seconds.")
e8 = tb1.energy_cons()

#Galaxies

galsys = creategalaxysystem(pctoau(15000.0))
print("\nGalaxy Rk4")
tstart = time.time()
galtraj = galsys.rk4(np.linspace(0, 10**10, 10000))
tend = time.time()
print(f"Time: {tend - tstart:.2f} seconds.")
e9 = galsys.energy_cons()

galsys = creategalaxysystem(pctoau(15000.0))
print("\nGalaxy Leapfrog")
tstart = time.time()
galtraj = galsys.leapfrog(np.linspace(0, 10**10, 10000))
tend = time.time()
print(f"Time {tend - tstart:.2f} seconds.")
e10 = galsys.energy_cons()